<span class="ͼ12">
<center><h1>ESMX_StartHere Tutorial</h1></center>
</span>

<span class="ͼ16">
<h2>Clean Previous Build and Run Files</h2>
</span>

This command will remove all build files, build directories, run files, and run logs.

In [ ]:
!make distclean

<span class="ͼ16">
<h2>Build Executable</h2>
</span>
This command will build and install the ESMX executable.

In [ ]:
!make

<span class="ͼ16">
<h2>Create a Run Configuration</h2>
</span>

Create a first draft for the `esmxRunTutorial.yaml` runtime configuration file. This configuration file uses the ESMX standard configuration format, which is yaml based. The following file will execute a ESMX Driver without components.

Once a esmxRunTutorial.yaml file exists `make runTutorial` can be used to execute `$(ESMF_INTERNAL_MPIRUN) -np $(NP) ./$(EXE) esmxRunTutorial.yaml`. By default, ESMF will log output to `PET##.ESMF_LogFile` logs.

In [ ]:
%%writefile esmxRunTutorial.yaml
ESMX:

  App:
    startTime:              2012-10-24T08:00:00
    stopTime:               2012-10-24T20:00:00

In [ ]:
!make runTutorial
!cat PET0.ESMF_LogFile

<span class="ͼ16">
<h2>Add Application Configuration Options</h2>
</span>

The second draft of the `esmxRunTutorial.yaml` file adds application configuration options to enable global resource control and control logging.

In [ ]:
%%writefile esmxRunTutorial.yaml
ESMX:

  App:
    globalResourceControl:  true
    logKindFlag:            ESMF_LOGKIND_Multi
    logAppendFlag:          false
    logFlush:               true
    startTime:              2012-10-24T08:00:00
    stopTime:               2012-10-24T20:00:00

In [ ]:
!make runTutorial
!cat PET0.ESMF_LogFile

<span class="ͼ16">
<h2>Add Driver Configuration Options</h2>
</span>

The third draft of the `esmxRunTutorial.yaml` file adds driver configuration options to control ESMX Driver output to ESMF logs.

In [ ]:
%%writefile esmxRunTutorial.yaml
ESMX:

  App:
    globalResourceControl:  true
    logKindFlag:            ESMF_LOGKIND_Multi
    logAppendFlag:          false
    logFlush:               true
    startTime:              2012-10-24T08:00:00
    stopTime:               2012-10-24T20:00:00

  Driver:
    attributes:
      Verbosity: low

In [ ]:
!make runTutorial
!cat PET0.ESMF_LogFile

<span class="ͼ16">
<h2>Add an ESMX_Data Component (ABC) to Configuration Options</h2>
</span>

The fourth draft of the `esmxRunTutorial.yaml` file adds a component to the ESMX Driver named `ABC`. The `ABC` component is then configured to execute an instance of the `ESMX_Data` model, which is included with ESMX. The configuration file does not contain a `runSequence`. In this case the driver will take one time step from startTime to stopTime. If the model has configured its own time step then it will take several time steps until the stopTime is reached. If the model has not configured its own time step then it will match the time step of the driver.

In [ ]:
%%writefile esmxRunTutorial.yaml
ESMX:

  App:
    globalResourceControl:  true
    logKindFlag:            ESMF_LOGKIND_Multi
    logAppendFlag:          false
    logFlush:               true
    startTime:              2012-10-24T08:00:00
    stopTime:               2012-10-24T20:00:00

  Driver:
    attributes:
      Verbosity: off
    componentList:  [ABC]

ABC:
  model:  ESMX_Data
  attributes:
    Verbosity: low

In [ ]:
!make runTutorial
!cat PET0.ESMF_LogFile

<span class="ͼ16">
<h2>Add a Second ESMX_Data Component (DEF) to Configuration Options</h2>
</span>

The fifth draft of the `esmxRunTutorial.yaml` file adds a second component to the ESMX Driver named `DEF`. The `DEF` component is then configured to  execute a second instance of the `ESMX_Data` model. The two models are configured to exchange data through NUOPC Connectors in the `runSequence` configuration option. Each `ESMX_Data` data model is individually configured with `importFields` and `exportFields`.

```yml
  importFields:
    <field_standard_name>: {dim: <dimensions>, min: <minimum valid value>, max: <maximum valid value>}
  exportFields:
    <field_standard_name>: {dim: <dimensions>, val: <prescribed fill value>}
```

In [ ]:
%%writefile esmxRunTutorial.yaml
ESMX:

  App:
    globalResourceControl:  true
    logKindFlag:            ESMF_LOGKIND_Multi
    logAppendFlag:          false
    logFlush:               true
    startTime:              2012-10-24T08:00:00
    stopTime:               2012-10-24T20:00:00

    ESMF_RUNTIME_PROFILE:         ON
    ESMF_RUNTIME_PROFILE_OUTPUT:  SUMMARY

  Driver:
    attributes:
      Verbosity: off
    componentList:  [ABC, DEF]

    runSequence: |
      @900
        ABC -> DEF
        DEF -> ABC
        ABC
        DEF
      @

ABC:
  model:  ESMX_Data
  attributes:
    Verbosity: high
  exportFields:
    sea_surface_temperature: {dim: 2, val: 273}
  output:
    write_final: false

DEF:
  model:  ESMX_Data
  attributes:
    Verbosity: high
  importFields:
    sea_surface_temperature: {dim: 2, min: 260, max: 280}
  output:
    write_final: false

In [ ]:
!make runTutorial
!cat PET0.ESMF_LogFile